<a href="https://colab.research.google.com/github/phamvanhung33182005-ux/Parameter_Identification_of_PMSM_using_PINN/blob/main/notebooks/03_baseline_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# ===== CELL 1: LOAD DATASET VÀ KIỂM TRA GPU =====

from google.colab import drive  # Import công cụ để kết nối Google Drive.
import os  # Import thư viện os để thao tác với đường dẫn và thư mục.
import numpy as np  # Import NumPy để đọc các file .npy.
import torch  # Import PyTorch để xây dựng và kiểm tra mô hình.

drive.mount('/content/drive')  # Gắn Google Drive vào môi trường Google Colab.

PROJECT_DIR = '/content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN'  # Khai báo thư mục gốc của project.

PROCESSED_DIR = os.path.join(PROJECT_DIR, 'dataset', 'processed')  # Tạo đường dẫn đến dataset đã preprocessing.

X_train = np.load(os.path.join(PROCESSED_DIR, 'X_train.npy'))  # Đọc input của tập training.

Y_train = np.load(os.path.join(PROCESSED_DIR, 'Y_train.npy'))  # Đọc target của tập training.

X_validation = np.load(os.path.join(PROCESSED_DIR, 'X_validation.npy'))  # Đọc input của tập validation.

Y_validation = np.load(os.path.join(PROCESSED_DIR, 'Y_validation.npy'))  # Đọc target của tập validation.

X_test = np.load(os.path.join(PROCESSED_DIR, 'X_test.npy'))  # Đọc input của tập test.

Y_test = np.load(os.path.join(PROCESSED_DIR, 'Y_test.npy'))  # Đọc target của tập test.

print('X_train:', X_train.shape)  # In kích thước input training.

print('Y_train:', Y_train.shape)  # In kích thước target training.

print('X_validation:', X_validation.shape)  # In kích thước input validation.

print('Y_validation:', Y_validation.shape)  # In kích thước target validation.

print('X_test:', X_test.shape)  # In kích thước input test.

print('Y_test:', Y_test.shape)  # In kích thước target test.

print('PyTorch version:', torch.__version__)  # In phiên bản PyTorch đang sử dụng.

print('CUDA available:', torch.cuda.is_available())  # Kiểm tra CUDA có khả dụng hay không.

if torch.cuda.is_available():  # Kiểm tra xem GPU CUDA có khả dụng hay không.
    print('GPU:', torch.cuda.get_device_name(0))  # In tên GPU nếu CUDA khả dụng.
else:  # Thực hiện nhánh này nếu CUDA không khả dụng.
    print('GPU: Không khả dụng, sử dụng CPU.')  # Thông báo mô hình sẽ chạy trên CPU.

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # Chọn GPU nếu có CUDA, ngược lại sử dụng CPU.

print('Device:', device)  # In thiết bị được chọn.

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
X_train: (90000, 5)
Y_train: (90000, 3)
X_validation: (30000, 5)
Y_validation: (30000, 3)
X_test: (30000, 5)
Y_test: (30000, 3)
PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
Device: cuda


In [8]:
# ===== CELL 2: KIỂM TRA INPUT/OUTPUT VÀ CHUYỂN SANG PYTORCH TENSOR =====

print('Số input features:', X_train.shape[1])  # Kiểm tra số lượng biến đầu vào.

print('Số output targets:', Y_train.shape[1])  # Kiểm tra số lượng tham số cần nhận dạng.

print('Input features: [ud, uq, Id, Iq, nm]')  # Ghi rõ 5 biến đầu vào của mô hình.

print('Output targets: [Rs, Ld, Lq]')  # Ghi rõ 3 tham số đầu ra của mô hình.

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)  # Chuyển input training từ NumPy sang FloatTensor.

Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)  # Chuyển target training sang FloatTensor.

X_validation_tensor = torch.tensor(X_validation, dtype=torch.float32)  # Chuyển input validation sang FloatTensor.

Y_validation_tensor = torch.tensor(Y_validation, dtype=torch.float32)  # Chuyển target validation sang FloatTensor.

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)  # Chuyển input test sang FloatTensor.

Y_test_tensor = torch.tensor(Y_test, dtype=torch.float32)  # Chuyển target test sang FloatTensor.

print('X_train_tensor:', X_train_tensor.shape)  # Kiểm tra kích thước Tensor input.

print('Y_train_tensor:', Y_train_tensor.shape)  # Kiểm tra kích thước Tensor target.

print('dtype:', X_train_tensor.dtype)  # Kiểm tra kiểu dữ liệu của Tensor.

print('Input sample:', X_train_tensor[0])  # In một mẫu input để kiểm tra dữ liệu.

print('Target sample:', Y_train_tensor[0])  # In target tương ứng để kiểm tra dữ liệu.

Số input features: 5
Số output targets: 3
Input features: [ud, uq, Id, Iq, nm]
Output targets: [Rs, Ld, Lq]
X_train_tensor: torch.Size([90000, 5])
Y_train_tensor: torch.Size([90000, 3])
dtype: torch.float32
Input sample: tensor([ 1.2205, -2.2199,  0.5614, -1.2761, -1.6356])
Target sample: tensor([-1.4635, -0.5408,  0.4599])


In [9]:
# ===== CELL 3: XÂY DỰNG KIẾN TRÚC BASELINE MLP =====

import torch.nn as nn  # Import module neural network của PyTorch.

class BaselineMLP(nn.Module):  # Khai báo lớp mạng MLP baseline kế thừa từ nn.Module.

    def __init__(self):  # Khởi tạo kiến trúc của mạng.
        super().__init__()  # Gọi hàm khởi tạo của lớp nn.Module.

        self.network = nn.Sequential(  # Tạo chuỗi các layer của MLP.
            nn.Linear(5, 64),  # Layer thứ nhất biến đổi 5 input features thành 64 neurons.
            nn.ReLU(),  # Hàm kích hoạt ReLU cho hidden layer thứ nhất.
            nn.Linear(64, 64),  # Layer thứ hai biến đổi 64 neurons thành 64 neurons.
            nn.ReLU(),  # Hàm kích hoạt ReLU cho hidden layer thứ hai.
            nn.Linear(64, 3)  # Output layer biến đổi 64 neurons thành 3 tham số Rs, Ld và Lq.
        )  # Kết thúc kiến trúc Sequential.

    def forward(self, x):  # Định nghĩa quá trình forward của mạng.
        return self.network(x)  # Đưa input qua toàn bộ mạng và trả về output.

model = BaselineMLP().to(device)  # Khởi tạo Baseline MLP và đưa model lên CPU hoặc GPU.

print(model)  # In toàn bộ kiến trúc của Baseline MLP.

BaselineMLP(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=3, bias=True)
  )
)


In [10]:
# ===== CELL 4: KIỂM TRA SỐ LƯỢNG THAM SỐ CỦA BASELINE MLP =====

total_parameters = sum(parameter.numel() for parameter in model.parameters())  # Tính tổng số parameter của toàn bộ mạng.

trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)  # Tính số parameter có thể được học.

print('Tổng số parameters:', total_parameters)  # In tổng số parameters của model.

print('Trainable parameters:', trainable_parameters)  # In số parameters sẽ được cập nhật trong quá trình training sau này.

Tổng số parameters: 4739
Trainable parameters: 4739


In [11]:
# ===== CELL 5: KIỂM TRA FORWARD PASS CỦA BASELINE MLP =====

model.eval()  # Đưa model về chế độ evaluation để kiểm tra forward pass.

sample_input = X_train_tensor[:32].to(device)  # Lấy 32 mẫu đầu tiên của tập training và đưa lên CPU/GPU.

with torch.no_grad():  # Tắt tính gradient vì cell này chỉ kiểm tra forward pass.
    sample_output = model(sample_input)  # Đưa batch input qua Baseline MLP.

print('Input shape:', sample_input.shape)  # In kích thước input đưa vào model.

print('Output shape:', sample_output.shape)  # In kích thước output của model.

print('Output sample:')  # In tiêu đề cho output mẫu.

print(sample_output[:5])  # In 5 kết quả dự đoán đầu tiên.

Input shape: torch.Size([32, 5])
Output shape: torch.Size([32, 3])
Output sample:
tensor([[0.0921, 0.1356, 0.0451],
        [0.0920, 0.1355, 0.0450],
        [0.0917, 0.1354, 0.0447],
        [0.0912, 0.1352, 0.0442],
        [0.0906, 0.1350, 0.0435]], device='cuda:0')


In [12]:
# ===== CELL 6: KIỂM TRA MODEL ĐANG CHẠY TRÊN DEVICE NÀO =====

first_parameter = next(model.parameters())  # Lấy parameter đầu tiên của model để kiểm tra device.

print('Model device:', first_parameter.device)  # In thiết bị đang lưu model.

print('Expected device:', device)  # In thiết bị mà chương trình đã lựa chọn.

Model device: cuda:0
Expected device: cuda
